# Nifty 50 Stock Market Analytics and Multi-Task Machine Learning Prediction System

**Internship Project** — IBM SkillsBuild Data Analytics with AI Academy Internship Program  
Conducted by **Bharat Care** in association with **AICTE**

---

> **Disclaimer:** This project is developed strictly for educational and research purposes. It does not constitute financial advice. All predictions are model estimates based on historical patterns and carry significant uncertainty. Past performance does not guarantee future results.

## 1. Project Introduction

This notebook presents a complete end-to-end data analytics and machine learning pipeline for Nifty 50 stock market data spanning 1999–2026. The workflow covers:
- 20-point data quality audit
- Robust data cleaning
- Comprehensive EDA (market, stock, sector, year, month, correlation)
- Technical indicator computation
- Lag and rolling feature engineering (no data leakage)
- Seven multi-task ML prediction objectives
- Chronological train/validation/test splitting
- Five models: Baseline, Linear Regression, Random Forest, XGBoost, LSTM
- Model explainability and central comparison table
- Saved models and cleaned data for the Streamlit dashboard

## 2. Objectives

1. Audit and clean the Nifty 50 historical dataset transparently.
2. Explore market-wide, sector, stock, year, and month-level patterns.
3. Compute reliable technical indicators grouped by ticker.
4. Engineer predictive features without look-ahead bias.
5. Predict: Next-Day Close, Return, Direction, High, Low, Volatility, and Market Regime.
6. Compare models fairly on chronological held-out test data.
7. Explain model predictions using feature importance.

## 3. Dataset Information

- **Source:** Yahoo Finance via Kaggle  
- **URL:** https://www.kaggle.com/datasets/kalyan197/nifty50-stocks1999-2026-daily-ohlcv-and-fundamentals  
- **Period:** 1999-01-01 to 2026-01-31  
- **Records:** ~287,310 rows across 49 stocks and 13 sectors  
- **Columns:** 25 (Date, Ticker, OHLCV, fundamentals, pre-computed indicators)

## 4. Import Libraries

In [ ]:
import os
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.inspection import permutation_importance
import xgboost as xgb
import joblib
from tqdm import tqdm

# Technical analysis
try:
    import ta
    TA_AVAILABLE = True
except ImportError:
    TA_AVAILABLE = False
    print('WARNING: ta library not found. Technical indicators will be computed manually.')

# LSTM (optional)
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    LSTM_AVAILABLE = True
    print(f'TensorFlow {tf.__version__} available — LSTM enabled.')
except ImportError:
    LSTM_AVAILABLE = False
    print('INFO: TensorFlow not available — LSTM model will be skipped.')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded successfully.')

## 5. Configuration

In [ ]:
# ─── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR        = Path('..')
DATA_DIR        = BASE_DIR / 'database'
MAIN_CSV        = DATA_DIR / 'nifty50_historical_data.csv'
SUMMARY_CSV     = DATA_DIR / 'nifty50_summary_statistics.csv'
META_JSON       = DATA_DIR / 'metadata.json'

MODEL_DIR       = Path('models')
SCALER_DIR      = MODEL_DIR / 'scalers'
OUTPUT_DIR      = Path('outputs')
FIGURES_DIR     = OUTPUT_DIR / 'figures'
METRICS_DIR     = OUTPUT_DIR / 'metrics'
CLEANED_DIR     = OUTPUT_DIR / 'cleaned_data'

for d in [MODEL_DIR, SCALER_DIR, OUTPUT_DIR, FIGURES_DIR, METRICS_DIR, CLEANED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ─── Fast-mode flag ───────────────────────────────────────────────────────────
# Set to True to train on a smaller subset for faster iteration
FAST_MODE = False
FAST_TICKERS = ['RELIANCE.NS', 'INFY.NS', 'HDFCBANK.NS', 'ICICIBANK.NS', 'TCS.NS']

# ─── Train/val/test split ratios ──────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# TEST_RATIO  = 0.15 (remainder)

# ─── Feature engineering ──────────────────────────────────────────────────────
LAG_PERIODS      = [1, 2, 3, 5, 10]
ROLLING_WINDOWS  = [5, 10, 20]

print('Configuration set.')
print(f'  FAST_MODE = {FAST_MODE}')
print(f'  MAIN_CSV  = {MAIN_CSV}')

## 6. Load Dataset

In [ ]:
# ─── Verify file existence ────────────────────────────────────────────────────
if not MAIN_CSV.exists():
    raise FileNotFoundError(
        f"Dataset not found at {MAIN_CSV}.\n"
        "Please download from: https://www.kaggle.com/datasets/kalyan197/"
        "nifty50-stocks1999-2026-daily-ohlcv-and-fundamentals\n"
        "and place nifty50_historical_data.csv in the database/ folder."
    )

# ─── Load metadata ────────────────────────────────────────────────────────────
if META_JSON.exists():
    with open(META_JSON) as f:
        metadata = json.load(f)
    print('Metadata loaded:')
    for k, v in metadata.items():
        if k != 'columns':
            print(f'  {k}: {v}')

# ─── Load main dataset ────────────────────────────────────────────────────────
print('\nLoading main dataset...')
df_raw = pd.read_csv(MAIN_CSV, low_memory=False)
print(f'Loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

# ─── Load summary statistics ──────────────────────────────────────────────────
if SUMMARY_CSV.exists():
    df_summary = pd.read_csv(SUMMARY_CSV)
    print(f'Summary statistics loaded: {df_summary.shape[0]} stocks')
else:
    df_summary = None
    print('Summary CSV not found — will be generated from main data.')

df_raw.head(3)

## 7. Data Quality Audit

This section implements a 20-point data quality checklist. Every finding is documented. No silent deletions occur here.

In [ ]:
print('=' * 70)
print('DATA QUALITY AUDIT — 20-POINT CHECKLIST')
print('=' * 70)

# 1. Dataset dimensions
print(f'\n[1] Dimensions: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')

# 2. Column names
print(f'\n[2] Columns ({len(df_raw.columns)}):')
print(list(df_raw.columns))

# 3. Data types
print('\n[3] Data types:')
print(df_raw.dtypes)

# 4. First and last records
print('\n[4a] First 3 records:')
display(df_raw.head(3))
print('\n[4b] Last 3 records:')
display(df_raw.tail(3))

In [ ]:
# 5. Descriptive statistics
print('\n[5] Descriptive Statistics:')
display(df_raw.describe(include='all').T)

In [ ]:
# 6. Missing values
print('\n[6] Missing Values:')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_report = pd.DataFrame({'Missing_Count': missing, 'Missing_%': missing_pct})
missing_report = missing_report[missing_report['Missing_Count'] > 0].sort_values('Missing_%', ascending=False)
if missing_report.empty:
    print('  No missing values found.')
else:
    display(missing_report)

# 7. Duplicate rows
n_dup_rows = df_raw.duplicated().sum()
print(f'\n[7] Fully duplicate rows: {n_dup_rows:,}')

# 8. Duplicate Date-Ticker combinations
n_dup_dt = df_raw.duplicated(subset=['Date', 'Ticker']).sum()
print(f'\n[8] Duplicate Date-Ticker combinations: {n_dup_dt:,}')

In [ ]:
# 9. Invalid dates
print('\n[9] Date inspection:')
try:
    dates_parsed = pd.to_datetime(df_raw['Date'], utc=True, errors='coerce')
    n_invalid_dates = dates_parsed.isna().sum()
    print(f'  Invalid/unparseable dates: {n_invalid_dates:,}')
    print(f'  Date range: {dates_parsed.min()} → {dates_parsed.max()}')
except Exception as e:
    print(f'  Date parsing error: {e}')

# 10. Impossible numerical values
print('\n[10] Impossible numerical values (negative prices):')
price_cols = ['Open', 'High', 'Low', 'Close']
for col in price_cols:
    if col in df_raw.columns:
        n_neg = (pd.to_numeric(df_raw[col], errors='coerce') <= 0).sum()
        if n_neg > 0:
            print(f'  {col}: {n_neg:,} zero/negative values')
        else:
            print(f'  {col}: OK (no zero/negative values)')

# 11. Volume checks
print('\n[11] Volume checks:')
if 'Volume' in df_raw.columns:
    n_neg_vol = (pd.to_numeric(df_raw['Volume'], errors='coerce') < 0).sum()
    n_zero_vol = (pd.to_numeric(df_raw['Volume'], errors='coerce') == 0).sum()
    print(f'  Negative volume: {n_neg_vol:,}')
    print(f'  Zero volume: {n_zero_vol:,}')

In [ ]:
# 12. OHLC consistency (High >= Low, High >= Open, High >= Close, Low <= Open, Low <= Close)
print('\n[12] OHLC consistency checks:')
num_df = df_raw[['Open','High','Low','Close']].apply(pd.to_numeric, errors='coerce')
n_hl = (num_df['High'] < num_df['Low']).sum()
n_ho = (num_df['High'] < num_df['Open']).sum()
n_hc = (num_df['High'] < num_df['Close']).sum()
n_lo = (num_df['Low'] > num_df['Open']).sum()
n_lc = (num_df['Low'] > num_df['Close']).sum()
print(f'  High < Low:   {n_hl:,}')
print(f'  High < Open:  {n_ho:,}')
print(f'  High < Close: {n_hc:,}')
print(f'  Low > Open:   {n_lo:,}')
print(f'  Low > Close:  {n_lc:,}')

# 13. Outlier inspection via IQR
print('\n[13] Outlier inspection (IQR method, Close price):')
close_num = pd.to_numeric(df_raw['Close'], errors='coerce').dropna()
Q1, Q3 = close_num.quantile(0.25), close_num.quantile(0.75)
IQR = Q3 - Q1
n_outliers = ((close_num < Q1 - 3*IQR) | (close_num > Q3 + 3*IQR)).sum()
print(f'  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
print(f'  Extreme outliers (3×IQR): {n_outliers:,} rows')
print('  NOTE: Extreme values are expected due to stock splits and long time range.')

In [ ]:
# 14. Ticker/Company/Sector consistency
print('\n[14] Ticker / Company / Sector consistency:')
print(f'  Unique tickers:  {df_raw["Ticker"].nunique()}')
print(f'  Unique companies: {df_raw["Company_Name"].nunique() if "Company_Name" in df_raw.columns else "N/A"}')
print(f'  Unique sectors:   {df_raw["Sector"].nunique() if "Sector" in df_raw.columns else "N/A"}')
if 'Sector' in df_raw.columns:
    print('  Sectors found:')
    print(df_raw['Sector'].value_counts())

# 15. Chronological ordering
print('\n[15] Chronological ordering check:')
dates_for_check = pd.to_datetime(df_raw['Date'], utc=True, errors='coerce')
is_sorted = dates_for_check.is_monotonic_increasing
print(f'  Globally sorted ascending: {is_sorted}')
print('  (Per-ticker ordering will be enforced during cleaning.)')

In [ ]:
# 16-17. Observations per stock and date range per stock
print('\n[16-17] Observations per stock and date ranges:')
stock_info = df_raw.groupby('Ticker').agg(
    Count=('Close', 'count'),
    First_Date=('Date', 'min'),
    Last_Date=('Date', 'max')
).reset_index().sort_values('Count', ascending=False)
display(stock_info.head(10))
print(f'  Min observations per stock: {stock_info["Count"].min()}')
print(f'  Max observations per stock: {stock_info["Count"].max()}')
print(f'  Mean observations per stock: {stock_info["Count"].mean():.0f}')

In [ ]:
# 18. Missing trading days check (using business days as proxy; note Indian holidays are not accounted for)
print('\n[18] Missing trading days — sample check on RELIANCE.NS:')
sample_ticker = 'RELIANCE.NS'
if sample_ticker in df_raw['Ticker'].values:
    rel_dates = pd.to_datetime(
        df_raw[df_raw['Ticker'] == sample_ticker]['Date'], utc=True, errors='coerce'
    ).dt.tz_convert('Asia/Kolkata').dt.normalize().dropna().sort_values()
    # Use business-day frequency as rough proxy (actual Indian market holidays not known)
    expected_bdays = pd.bdate_range(rel_dates.min(), rel_dates.max())
    missing_bdays = expected_bdays.difference(rel_dates)
    print(f'  Actual trading days: {len(rel_dates)}')
    print(f'  Business days (Mon-Fri) in range: {len(expected_bdays)}')
    print(f'  Apparent missing business days: {len(missing_bdays)}')
    print('  NOTE: Many of these are Indian public/market holidays — not data errors.')

# 19. Look-ahead / leakage-prone columns
print('\n[19] Look-ahead / leakage risk assessment:')
leakage_risk = {
    '52Week_High': 'Contains future information within the rolling 52-week window for recent dates.',
    '52Week_Low':  'Contains future information within the rolling 52-week window for recent dates.',
    'MA_50':       'Pre-computed; will be recomputed per-ticker to ensure correctness.',
    'MA_200':      'Pre-computed; will be recomputed per-ticker to ensure correctness.',
    'Volatility_20D': 'Pre-computed; will be recomputed per-ticker.',
    'Daily_Return':   'Pre-computed daily return — verify it is (Close_t / Close_{t-1} - 1).',
    'Market_Cap':     'Appears static/snapshot — treat as fundamental feature, not time-varying.',
    'PE_Ratio':       'Appears static/snapshot — same caveat.',
}
for col, note in leakage_risk.items():
    if col in df_raw.columns:
        print(f'  [{col}]: {note}')

print('\n  DECISION: 52Week_High and 52Week_Low will be EXCLUDED from ML features.')
print('  Pre-computed indicators will be RECOMPUTED from OHLCV per ticker to ensure no leakage.')

# 20. Data quality summary
print('\n' + '=' * 70)
print('[20] DATA QUALITY REPORT SUMMARY')
print('=' * 70)
quality_issues = [
    ('Fully duplicate rows', n_dup_rows),
    ('Duplicate Date-Ticker', n_dup_dt),
    ('Invalid dates', n_invalid_dates),
    ('OHLC violations (High<Low)', n_hl),
    ('Missing Close values', int(missing.get('Close', 0))),
    ('Zero/negative Close', int((pd.to_numeric(df_raw['Close'], errors='coerce') <= 0).sum())),
]
for issue, count in quality_issues:
    status = '✓ OK' if count == 0 else f'⚠ {count:,} issues'
    print(f'  {issue:<40} {status}')
print('\nConclusion: Dataset is largely clean. Cleaning steps will address all findings.')

## 8. Data Cleaning

Every cleaning decision is documented. No data is silently deleted.

In [ ]:
print('Starting data cleaning pipeline...')
df = df_raw.copy()
n_start = len(df)

# Step 1: Parse dates with timezone → normalize to date only
# The raw dates include timezone offset (+05:30). We parse and convert to IST.
print('\nStep 1: Date parsing...')
df['Date'] = pd.to_datetime(df['Date'], utc=True, errors='coerce').dt.tz_convert('Asia/Kolkata').dt.normalize()
n_invalid = df['Date'].isna().sum()
if n_invalid > 0:
    print(f'  Dropping {n_invalid:,} rows with unparseable dates.')
    df = df.dropna(subset=['Date'])
else:
    print('  All dates parsed successfully.')

# Step 2: Sort by Ticker then Date
print('\nStep 2: Sorting by Ticker and Date...')
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
print('  Done.')

# Step 3: Remove fully duplicate rows
print('\nStep 3: Removing duplicate rows...')
n_before = len(df)
df = df.drop_duplicates()
print(f'  Removed: {n_before - len(df):,} fully duplicate rows.')

# Step 4: Remove duplicate Date-Ticker (keep first)
print('\nStep 4: Removing duplicate Date-Ticker pairs (keep first)...')
n_before = len(df)
df = df.drop_duplicates(subset=['Date', 'Ticker'], keep='first')
print(f'  Removed: {n_before - len(df):,} duplicate Date-Ticker rows.')

# Step 5: Numeric type conversion for OHLCV and fundamentals
print('\nStep 5: Numeric type conversion...')
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Dividend', 'Stock_Split',
                'Daily_Return', 'Volatility_20D', 'MA_50', 'MA_200',
                'Market_Cap', 'PE_Ratio', 'Forward_PE', 'PEG_Ratio',
                'Price_to_Book', 'Dividend_Yield', 'EPS', 'Beta',
                '52Week_High', '52Week_Low']
numeric_cols = [c for c in numeric_cols if c in df.columns]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print(f'  Converted {len(numeric_cols)} columns to numeric.')

print(f'\nAfter basic cleaning: {len(df):,} rows (removed {n_start - len(df):,} total).')

In [ ]:
# Step 6: Handle missing OHLCV values
print('Step 6: Missing OHLCV handling...')
print('  Decision: Forward-fill within each ticker group for price columns.')
print('  Rationale: In financial time series, the previous trading day close is the')
print('  best proxy for a missing price. Mean imputation across all stocks would')
print('  introduce cross-stock contamination.')

price_fill_cols = ['Open', 'High', 'Low', 'Close']
before_na = df[price_fill_cols].isna().sum()
df[price_fill_cols] = df.groupby('Ticker')[price_fill_cols].transform(
    lambda x: x.ffill()
)
after_na = df[price_fill_cols].isna().sum()
print('  Missing prices before / after forward-fill:')
for c in price_fill_cols:
    print(f'    {c}: {int(before_na[c])} → {int(after_na[c])}')

# Volume: fill with 0 where missing (trading halt / data gap)
n_vol_na = df['Volume'].isna().sum()
df['Volume'] = df['Volume'].fillna(0)
print(f'  Volume NaN filled with 0: {n_vol_na:,} rows')

# Step 7: Drop rows where Close is still NaN (first row of each ticker if no prior data)
n_before = len(df)
df = df.dropna(subset=['Close'])
print(f'\nStep 7: Dropped {n_before - len(df):,} rows where Close remained NaN after forward-fill.')

# Step 8: Drop rows with zero/negative Close
print('\nStep 8: Removing rows with zero/negative Close...')
n_before = len(df)
df = df[df['Close'] > 0]
print(f'  Removed: {n_before - len(df):,} rows.')

# Step 9: Categorical cleaning
print('\nStep 9: Categorical cleaning...')
for col in ['Ticker', 'Company_Name', 'Sector']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
print('  Ticker, Company_Name, Sector stripped of whitespace.')

print(f'\nFinal cleaned dataset: {len(df):,} rows × {df.shape[1]} columns.')
print(f'Tickers retained: {df["Ticker"].nunique()}')

In [ ]:
# Step 10: Recompute Daily_Return from Close prices (group-wise, no cross-ticker contamination)
print('Step 10: Recomputing Daily_Return per ticker from Close prices...')
print('  Rationale: Ensures the pre-computed column matches actual OHLCV data.')
df['Daily_Return'] = df.groupby('Ticker')['Close'].pct_change()
print('  Done. First value per ticker is NaN (expected for pct_change).')

# Step 11: Fundamental columns — fill NaN with group median (ticker-wise)
print('\nStep 11: Filling fundamental NaN values with ticker-wise median...')
print('  Decision: Fundamental data (PE_Ratio, EPS, etc.) changes slowly.')
print('  Ticker-wise median is preferred over global mean to preserve cross-stock differences.')
fundamental_cols = ['PE_Ratio', 'Forward_PE', 'PEG_Ratio', 'Price_to_Book',
                    'Dividend_Yield', 'EPS', 'Beta', 'Market_Cap']
fundamental_cols = [c for c in fundamental_cols if c in df.columns]
for col in fundamental_cols:
    df[col] = df.groupby('Ticker')[col].transform(lambda x: x.fillna(x.median()))
print(f'  Processed {len(fundamental_cols)} fundamental columns.')

# Step 12: Fast-mode subsetting
if FAST_MODE:
    print('\n⚡ FAST_MODE enabled — subsetting to', FAST_TICKERS)
    df = df[df['Ticker'].isin(FAST_TICKERS)].reset_index(drop=True)
    print(f'  Subset size: {len(df):,} rows')

# Save cleaned dataset
clean_path = CLEANED_DIR / 'nifty50_cleaned.csv'
df.to_csv(clean_path, index=False)
print(f'\nCleaned data saved to: {clean_path}')
print(f'Final shape: {df.shape}')
df.head(3)

## 9. Exploratory Data Analysis

### 9A. Overall Market Analysis

In [ ]:
print('=== OVERALL MARKET ANALYSIS ===')
print(f'Companies: {df["Ticker"].nunique()}')
print(f'Sectors:   {df["Sector"].nunique()}')
print(f'Records:   {len(df):,}')
print(f'Date range: {df["Date"].min().date()} → {df["Date"].max().date()}')

# Market-wide daily return distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].hist(df['Daily_Return'].dropna(), bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Market-Wide Daily Return Distribution', fontsize=13)
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Frequency')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.7, label='Zero return')
axes[0].legend()

# Avg close per year (all stocks)
df['Year'] = df['Date'].dt.year
yearly_avg = df.groupby('Year')['Close'].median()
axes[1].plot(yearly_avg.index, yearly_avg.values, marker='o', color='navy', linewidth=2)
axes[1].set_title('Median Close Price Across All Stocks — Yearly', fontsize=13)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Median Close Price (₹)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_market_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {FIGURES_DIR}/eda_market_overview.png')

### 9B. Individual Stock Analysis

In [ ]:
def plot_stock_analysis(ticker, df):
    """Plot price, volume, returns and moving averages for a given ticker."""
    s = df[df['Ticker'] == ticker].copy().sort_values('Date')
    if s.empty:
        print(f'No data for {ticker}')
        return
    fig, axes = plt.subplots(4, 1, figsize=(16, 16), sharex=True)

    # Price + MA
    axes[0].plot(s['Date'], s['Close'], color='navy', linewidth=1, label='Close')
    if 'MA_50' in s.columns:
        axes[0].plot(s['Date'], s['MA_50'], color='orange', linewidth=1, alpha=0.8, label='MA50')
    if 'MA_200' in s.columns:
        axes[0].plot(s['Date'], s['MA_200'], color='red', linewidth=1, alpha=0.8, label='MA200')
    axes[0].set_title(f'{ticker} — Price History', fontsize=13)
    axes[0].set_ylabel('Price (₹)')
    axes[0].legend()

    # Volume
    axes[1].bar(s['Date'], s['Volume'], color='steelblue', alpha=0.6)
    axes[1].set_title('Volume')
    axes[1].set_ylabel('Volume')

    # Daily return
    ret_color = ['green' if r >= 0 else 'red' for r in s['Daily_Return'].fillna(0)]
    axes[2].bar(s['Date'], s['Daily_Return'], color=ret_color, alpha=0.7)
    axes[2].set_title('Daily Return')
    axes[2].set_ylabel('Return')
    axes[2].axhline(0, color='black', linewidth=0.5)

    # Rolling volatility
    roll_vol = s['Daily_Return'].rolling(20).std() * np.sqrt(252)
    axes[3].plot(s['Date'], roll_vol, color='purple', linewidth=1)
    axes[3].set_title('20-Day Rolling Annualized Volatility')
    axes[3].set_ylabel('Volatility')
    axes[3].set_xlabel('Date')

    plt.suptitle(f'{ticker} — Full Analysis', fontsize=15, y=1.01, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'stock_{ticker.replace(".","_")}.png', dpi=100, bbox_inches='tight')
    plt.show()

# Demonstrate for RELIANCE.NS
plot_stock_analysis('RELIANCE.NS', df)

### 9C. Sector Analysis

In [ ]:
sector_stats = df.groupby('Sector').agg(
    Companies=('Ticker', 'nunique'),
    Avg_Daily_Return=('Daily_Return', 'mean'),
    Avg_Volatility=('Daily_Return', lambda x: x.std() * np.sqrt(252)),
    Avg_MarketCap=('Market_Cap', 'median')
).reset_index().sort_values('Avg_Daily_Return', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
# Avg daily return by sector
colors_ret = ['green' if v > 0 else 'red' for v in sector_stats['Avg_Daily_Return']]
axes[0].barh(sector_stats['Sector'], sector_stats['Avg_Daily_Return'] * 100, color=colors_ret)
axes[0].set_title('Avg Daily Return by Sector (%)', fontsize=12)
axes[0].set_xlabel('Avg Daily Return (%)')
axes[0].axvline(0, color='black', linewidth=0.8)

# Volatility by sector
axes[1].barh(sector_stats['Sector'], sector_stats['Avg_Volatility'] * 100, color='steelblue')
axes[1].set_title('Annualized Volatility by Sector (%)', fontsize=12)
axes[1].set_xlabel('Volatility (%)')

# Companies per sector
axes[2].barh(sector_stats['Sector'], sector_stats['Companies'], color='teal')
axes[2].set_title('Number of Companies per Sector', fontsize=12)
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_sector_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print(sector_stats.to_string(index=False))

### 9D. Yearly Analysis

In [ ]:
def yearly_analysis(ticker, df):
    s = df[df['Ticker'] == ticker].copy().sort_values('Date')
    s['Year'] = s['Date'].dt.year
    yearly = s.groupby('Year').agg(
        Open=('Open', 'first'),
        Close=('Close', 'last'),
        High=('High', 'max'),
        Low=('Low', 'min'),
        Avg_Daily_Return=('Daily_Return', 'mean'),
        Volatility=('Daily_Return', 'std'),
        Total_Volume=('Volume', 'sum'),
        Trading_Days=('Close', 'count')
    ).reset_index()
    yearly['Annual_Return_%'] = ((yearly['Close'] - yearly['Open']) / yearly['Open'] * 100).round(2)
    yearly['Volatility_Ann_%'] = (yearly['Volatility'] * np.sqrt(252) * 100).round(2)
    return yearly

yr_df = yearly_analysis('INFY.NS', df)
print('INFY.NS — Yearly Analysis:')
display(yr_df[['Year','Open','Close','High','Low','Annual_Return_%','Volatility_Ann_%','Total_Volume']].tail(15))

fig, ax = plt.subplots(figsize=(16, 5))
bar_colors = ['green' if v > 0 else 'red' for v in yr_df['Annual_Return_%']]
ax.bar(yr_df['Year'], yr_df['Annual_Return_%'], color=bar_colors, alpha=0.8, edgecolor='white')
ax.set_title('INFY.NS — Annual Return (%)', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Annual Return (%)')
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_yearly_INFY.png', dpi=120, bbox_inches='tight')
plt.show()

### 9E. Monthly Analysis

In [ ]:
df['Month'] = df['Date'].dt.month
monthly_market = df.groupby('Month')['Daily_Return'].mean() * 100

fig, ax = plt.subplots(figsize=(14, 5))
bar_colors = ['green' if v > 0 else 'red' for v in monthly_market]
ax.bar(monthly_market.index, monthly_market.values, color=bar_colors, alpha=0.8, edgecolor='white')
ax.set_title('Average Daily Return by Month — All Stocks (Seasonal Pattern)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Avg Daily Return (%)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_monthly_seasonality.png', dpi=120, bbox_inches='tight')
plt.show()

### 9F. Correlation Analysis

In [ ]:
# Correlation of numeric features
corr_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Daily_Return',
             'Volatility_20D', 'MA_50', 'MA_200', 'PE_Ratio', 'Beta', 'EPS']
corr_cols = [c for c in corr_cols if c in df.columns]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('NOTE: Correlation measures linear association only and does not imply causation.')

## 10. Feature Engineering

All features use only information available at or before the prediction time (no look-ahead bias).

In [ ]:
print('Computing feature engineering...')
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

def compute_features(group):
    g = group.copy()
    close = g['Close']
    volume = g['Volume']
    high = g['High']
    low = g['Low']

    # Lag features
    for lag in LAG_PERIODS:
        g[f'Close_Lag{lag}'] = close.shift(lag)
        g[f'Return_Lag{lag}'] = close.pct_change(lag)

    # Rolling statistics
    for win in ROLLING_WINDOWS:
        g[f'SMA_{win}'] = close.rolling(win).mean()
        g[f'STD_{win}'] = close.rolling(win).std()
        g[f'Vol_{win}'] = close.pct_change().rolling(win).std() * np.sqrt(252)

    # Price momentum
    g['Momentum_5']  = close - close.shift(5)
    g['Momentum_10'] = close - close.shift(10)

    # High-Low range
    g['HL_Range']     = high - low
    g['HL_Range_Pct'] = (high - low) / close.shift(1).replace(0, np.nan)

    # Price relative to moving averages (from dataset, recomputed)
    sma50  = close.rolling(50).mean()
    sma200 = close.rolling(200).mean()
    g['Price_vs_SMA50']  = (close - sma50) / sma50.replace(0, np.nan)
    g['Price_vs_SMA200'] = (close - sma200) / sma200.replace(0, np.nan)

    # Volume change
    g['Volume_Change'] = volume.pct_change()
    g['Volume_SMA5']   = volume.rolling(5).mean()

    return g

print('  Applying feature engineering per ticker (this may take a moment)...')
df = df.groupby('Ticker', group_keys=False).apply(compute_features)
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
print(f'  Feature engineering complete. Shape: {df.shape}')

## 11. Technical Indicators

Computed per ticker using the `ta` library or manual implementations.

In [ ]:
print('Computing technical indicators per ticker...')

def add_technical_indicators(group):
    g = group.copy().sort_values('Date')
    close = g['Close']
    high  = g['High']
    low   = g['Low']
    vol   = g['Volume']

    if TA_AVAILABLE and len(g) >= 200:
        # Moving averages
        g['SMA_20']   = ta.trend.sma_indicator(close, window=20)
        g['SMA_50']   = ta.trend.sma_indicator(close, window=50)
        g['SMA_100']  = ta.trend.sma_indicator(close, window=100)
        g['SMA_200']  = ta.trend.sma_indicator(close, window=200)
        g['EMA_20']   = ta.trend.ema_indicator(close, window=20)
        g['EMA_50']   = ta.trend.ema_indicator(close, window=50)
        # RSI
        g['RSI_14']   = ta.momentum.RSIIndicator(close, window=14).rsi()
        # MACD
        macd_ind      = ta.trend.MACD(close)
        g['MACD']     = macd_ind.macd()
        g['MACD_Sig'] = macd_ind.macd_signal()
        g['MACD_Diff']= macd_ind.macd_diff()
        # Bollinger Bands
        bb            = ta.volatility.BollingerBands(close, window=20)
        g['BB_High']  = bb.bollinger_hband()
        g['BB_Low']   = bb.bollinger_lband()
        g['BB_Mid']   = bb.bollinger_mavg()
        g['BB_Width'] = (g['BB_High'] - g['BB_Low']) / g['BB_Mid'].replace(0, np.nan)
        # ATR
        g['ATR_14']   = ta.volatility.AverageTrueRange(high, low, close, window=14).average_true_range()
    else:
        # Manual fallback
        g['SMA_20']   = close.rolling(20).mean()
        g['SMA_50']   = close.rolling(50).mean()
        g['SMA_100']  = close.rolling(100).mean()
        g['SMA_200']  = close.rolling(200).mean()
        g['EMA_20']   = close.ewm(span=20, adjust=False).mean()
        g['EMA_50']   = close.ewm(span=50, adjust=False).mean()
        # RSI manual
        delta = close.diff()
        gain  = delta.clip(lower=0).rolling(14).mean()
        loss  = (-delta.clip(upper=0)).rolling(14).mean()
        rs    = gain / loss.replace(0, np.nan)
        g['RSI_14']   = 100 - 100 / (1 + rs)
        # MACD manual
        ema12         = close.ewm(span=12, adjust=False).mean()
        ema26         = close.ewm(span=26, adjust=False).mean()
        g['MACD']     = ema12 - ema26
        g['MACD_Sig'] = g['MACD'].ewm(span=9, adjust=False).mean()
        g['MACD_Diff']= g['MACD'] - g['MACD_Sig']
        # Bollinger Bands manual
        sma20         = close.rolling(20).mean()
        std20         = close.rolling(20).std()
        g['BB_High']  = sma20 + 2 * std20
        g['BB_Low']   = sma20 - 2 * std20
        g['BB_Mid']   = sma20
        g['BB_Width'] = (g['BB_High'] - g['BB_Low']) / sma20.replace(0, np.nan)
        # ATR manual
        tr1 = high - low
        tr2 = (high - close.shift(1)).abs()
        tr3 = (low  - close.shift(1)).abs()
        tr  = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        g['ATR_14'] = tr.rolling(14).mean()

    return g

df = df.groupby('Ticker', group_keys=False).apply(add_technical_indicators)
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
print(f'Technical indicators added. Shape: {df.shape}')

## 12. Target Creation

**Critical:** All targets are created by **shifting future values**. Features use only past/current data. This prevents data leakage.

In [ ]:
print('Creating prediction targets (future-shifted)...')
print('  Method: df.groupby(Ticker)[col].shift(-1)')
print('  This assigns the NEXT row\'s value as the target for the current row.')
print('  The last row per ticker will have NaN targets and is dropped before training.')

df['Next_Day_Close']      = df.groupby('Ticker')['Close'].shift(-1)
df['Next_Day_High']       = df.groupby('Ticker')['High'].shift(-1)
df['Next_Day_Low']        = df.groupby('Ticker')['Low'].shift(-1)
df['Next_Day_Return']     = df.groupby('Ticker')['Daily_Return'].shift(-1)

# Rolling volatility (20-day annualized) target
df['Roll_Vol_20']         = df.groupby('Ticker')['Daily_Return'].transform(
    lambda x: x.rolling(20).std() * np.sqrt(252)
)
df['Next_Day_Volatility'] = df.groupby('Ticker')['Roll_Vol_20'].shift(-1)

# Direction: 1 = next-day close > current close, 0 = next-day close <= current close
df['Next_Day_Direction'] = (df['Next_Day_Close'] > df['Close']).astype(float)

# Market regime based on TRAINING-DATA quantiles (will be finalized after split)
# We use a temporary label here; thresholds are set from training data only
vol_quantiles = df['Roll_Vol_20'].quantile([0.33, 0.67])
def assign_regime(v):
    if pd.isna(v): return np.nan
    if v <= vol_quantiles[0.33]: return 0   # Low volatility
    elif v <= vol_quantiles[0.67]: return 1  # Medium volatility
    else: return 2                           # High volatility

df['Market_Regime'] = df['Roll_Vol_20'].apply(assign_regime)
df['Next_Day_Regime'] = df.groupby('Ticker')['Market_Regime'].shift(-1)

print('  Targets created:')
target_cols = ['Next_Day_Close','Next_Day_High','Next_Day_Low','Next_Day_Return',
               'Next_Day_Volatility','Next_Day_Direction','Next_Day_Regime']
for t in target_cols:
    print(f'    {t}: {df[t].notna().sum():,} non-null values')

## 13. Train / Validation / Test Split

**Why NOT random splitting?**  
In time-series data, randomly shuffling rows before splitting would allow training data to contain observations from dates *after* the test observations. The model would effectively have access to future information during training — this is data leakage. For fair evaluation, the test set must consist of the most recent observations that the model has never seen.

We use a **chronological 70/15/15 split** based on the global date axis.

In [ ]:
# ─── Define ML feature set (no leakage columns) ───────────────────────────────
EXCLUDE_COLS = [
    'Date', 'Ticker', 'Company_Name', 'Sector',
    'Next_Day_Close', 'Next_Day_High', 'Next_Day_Low', 'Next_Day_Return',
    'Next_Day_Volatility', 'Next_Day_Direction', 'Next_Day_Regime', 'Market_Regime',
    '52Week_High', '52Week_Low',   # potential look-ahead
    'Roll_Vol_20', 'Year', 'Month',
    'MA_50', 'MA_200',             # use recomputed SMA_50, SMA_200 instead
    'Volatility_20D',              # use recomputed Vol_20
    'Daily_Return',                # recomputed as feature via lags
]

FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]
print(f'Feature columns ({len(FEATURE_COLS)}):')
print(FEATURE_COLS)

# ─── Chronological date split ─────────────────────────────────────────────────
df_ml = df.dropna(subset=['Next_Day_Close'] + FEATURE_COLS).copy()
print(f'\nML-ready rows (after dropping NaN): {len(df_ml):,}')

all_dates = df_ml['Date'].sort_values().unique()
n_dates   = len(all_dates)
train_cut = all_dates[int(n_dates * TRAIN_RATIO)]
val_cut   = all_dates[int(n_dates * (TRAIN_RATIO + VAL_RATIO))]

train_df  = df_ml[df_ml['Date'] <= train_cut].copy()
val_df    = df_ml[(df_ml['Date'] > train_cut) & (df_ml['Date'] <= val_cut)].copy()
test_df   = df_ml[df_ml['Date'] > val_cut].copy()

print(f'\nChronological split (by unique dates):')
print(f'  Train: {train_df["Date"].min().date()} → {train_df["Date"].max().date()}  ({len(train_df):,} rows)')
print(f'  Val:   {val_df["Date"].min().date()}  → {val_df["Date"].max().date()}  ({len(val_df):,} rows)')
print(f'  Test:  {test_df["Date"].min().date()}  → {test_df["Date"].max().date()}  ({len(test_df):,} rows)')

In [ ]:
# Prepare X/y matrices
X_train = train_df[FEATURE_COLS].select_dtypes(include=[np.number]).fillna(0)
X_val   = val_df[FEATURE_COLS].select_dtypes(include=[np.number]).fillna(0)
X_test  = test_df[FEATURE_COLS].select_dtypes(include=[np.number]).fillna(0)

# Align columns (after select_dtypes some cols might differ)
FEATURE_COLS_FINAL = list(X_train.columns)
X_val  = X_val[FEATURE_COLS_FINAL]
X_test = X_test[FEATURE_COLS_FINAL]

# Targets
y_close_train  = train_df['Next_Day_Close']
y_close_val    = val_df['Next_Day_Close']
y_close_test   = test_df['Next_Day_Close']

y_ret_train    = train_df['Next_Day_Return'].dropna()
y_ret_test     = test_df['Next_Day_Return'].dropna()

y_dir_train    = train_df['Next_Day_Direction']
y_dir_test     = test_df['Next_Day_Direction']

y_high_train   = train_df['Next_Day_High']
y_high_test    = test_df['Next_Day_High']

y_low_train    = train_df['Next_Day_Low']
y_low_test     = test_df['Next_Day_Low']

y_vol_train    = train_df['Next_Day_Volatility'].dropna()
y_vol_test     = test_df['Next_Day_Volatility'].dropna()

y_regime_train = train_df['Next_Day_Regime'].dropna()
y_regime_test  = test_df['Next_Day_Regime'].dropna()

# Scale features for linear model
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)
joblib.dump(scaler, SCALER_DIR / 'feature_scaler.joblib')
print('Feature scaler saved.')

# Target scaler for close price (for LSTM)
target_scaler = StandardScaler()
target_scaler.fit(y_close_train.values.reshape(-1, 1))
joblib.dump(target_scaler, SCALER_DIR / 'target_scaler.joblib')
print('Target scaler saved.')
print(f'\nX_train shape: {X_train.shape}, X_test shape: {X_test.shape}')

## 14. Baseline Model

The naive baseline predicts the **previous day's close** as tomorrow's close, and the **previous day's return sign** as tomorrow's direction.

In [ ]:
results = {}  # Central model comparison store

# Baseline: previous close = predicted next close
baseline_pred_close = test_df['Close'].values
baseline_pred_dir   = (test_df['Close'].values > test_df['Close_Lag1'].values).astype(float)

y_close_test_arr = y_close_test.values
y_dir_test_arr   = y_dir_test.values

mae_b  = mean_absolute_error(y_close_test_arr, baseline_pred_close)
rmse_b = np.sqrt(mean_squared_error(y_close_test_arr, baseline_pred_close))
r2_b   = r2_score(y_close_test_arr, baseline_pred_close)
acc_b  = accuracy_score(y_dir_test_arr, baseline_pred_dir)
f1_b   = f1_score(y_dir_test_arr, baseline_pred_dir, zero_division=0)

results['Baseline'] = {'Task': 'Close', 'MAE': mae_b, 'RMSE': rmse_b, 'R2': r2_b,
                        'Dir_Accuracy': acc_b, 'Dir_F1': f1_b}

print('Baseline Model Results:')
print(f'  Close Prediction — MAE: {mae_b:.4f}, RMSE: {rmse_b:.4f}, R²: {r2_b:.4f}')
print(f'  Direction        — Accuracy: {acc_b:.4f}, F1: {f1_b:.4f}')

## 15. Linear Regression

In [ ]:
print('Training Linear Regression for Next_Day_Close...')
lr_model = LinearRegression()
lr_model.fit(X_train_sc, y_close_train)
lr_pred = lr_model.predict(X_test_sc)

mae_lr  = mean_absolute_error(y_close_test, lr_pred)
rmse_lr = np.sqrt(mean_squared_error(y_close_test, lr_pred))
r2_lr   = r2_score(y_close_test, lr_pred)

results['Linear Regression'] = {'Task': 'Close', 'MAE': mae_lr, 'RMSE': rmse_lr, 'R2': r2_lr}
print(f'  MAE: {mae_lr:.4f}  RMSE: {rmse_lr:.4f}  R²: {r2_lr:.4f}')

# Also train for direction using Logistic Regression
print('Training Logistic Regression for Next_Day_Direction...')
from sklearn.linear_model import LogisticRegression
lgr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lgr_model.fit(X_train_sc, y_dir_train)
lgr_pred = lgr_model.predict(X_test_sc)
lgr_prob = lgr_model.predict_proba(X_test_sc)[:, 1]

acc_lgr = accuracy_score(y_dir_test, lgr_pred)
f1_lgr  = f1_score(y_dir_test, lgr_pred, zero_division=0)
auc_lgr = roc_auc_score(y_dir_test, lgr_prob)
results['Logistic Regression'] = {'Task': 'Direction', 'Dir_Accuracy': acc_lgr,
                                    'Dir_F1': f1_lgr, 'ROC_AUC': auc_lgr}
print(f'  Accuracy: {acc_lgr:.4f}  F1: {f1_lgr:.4f}  AUC: {auc_lgr:.4f}')

## 16. Random Forest

In [ ]:
# ── Random Forest Regressor (Next_Day_Close) ──
print('Training Random Forest Regressor (Next_Day_Close)...')
n_est = 50 if FAST_MODE else 200
rf_reg = RandomForestRegressor(
    n_estimators=n_est, max_depth=12, min_samples_leaf=5,
    n_jobs=-1, random_state=RANDOM_STATE
)
rf_reg.fit(X_train, y_close_train)
rf_pred = rf_reg.predict(X_test)

mae_rf  = mean_absolute_error(y_close_test, rf_pred)
rmse_rf = np.sqrt(mean_squared_error(y_close_test, rf_pred))
r2_rf   = r2_score(y_close_test, rf_pred)
results['Random Forest'] = {'Task': 'Close', 'MAE': mae_rf, 'RMSE': rmse_rf, 'R2': r2_rf}
print(f'  MAE: {mae_rf:.4f}  RMSE: {rmse_rf:.4f}  R²: {r2_rf:.4f}')
joblib.dump(rf_reg, MODEL_DIR / 'rf_close_model.joblib')

# ── Random Forest Classifier (Next_Day_Direction) ──
print('\nTraining Random Forest Classifier (Next_Day_Direction)...')
rf_clf = RandomForestClassifier(
    n_estimators=n_est, max_depth=10, min_samples_leaf=10,
    n_jobs=-1, random_state=RANDOM_STATE
)
rf_clf.fit(X_train, y_dir_train)
rf_dir_pred = rf_clf.predict(X_test)
rf_dir_prob = rf_clf.predict_proba(X_test)[:, 1]

acc_rf  = accuracy_score(y_dir_test, rf_dir_pred)
f1_rf   = f1_score(y_dir_test, rf_dir_pred, zero_division=0)
auc_rf  = roc_auc_score(y_dir_test, rf_dir_prob)
results['Random Forest Classifier'] = {'Task': 'Direction', 'Dir_Accuracy': acc_rf,
                                         'Dir_F1': f1_rf, 'ROC_AUC': auc_rf}
print(f'  Accuracy: {acc_rf:.4f}  F1: {f1_rf:.4f}  AUC: {auc_rf:.4f}')
joblib.dump(rf_clf, MODEL_DIR / 'rf_direction_model.joblib')
print('Models saved.')

In [ ]:
# RF for High, Low, Return, Volatility
for target_name, y_tr, y_te, model_name in [
    ('Next_Day_High',       y_high_train, y_high_test, 'rf_high'),
    ('Next_Day_Low',        y_low_train,  y_low_test,  'rf_low'),
]:
    print(f'Training RF Regressor for {target_name}...')
    m = RandomForestRegressor(
        n_estimators=n_est, max_depth=12, min_samples_leaf=5,
        n_jobs=-1, random_state=RANDOM_STATE
    )
    m.fit(X_train, y_tr)
    pred = m.predict(X_test)
    mae_  = mean_absolute_error(y_te, pred)
    rmse_ = np.sqrt(mean_squared_error(y_te, pred))
    r2_   = r2_score(y_te, pred)
    results[f'RF_{target_name}'] = {'Task': target_name, 'MAE': mae_, 'RMSE': rmse_, 'R2': r2_}
    print(f'  MAE: {mae_:.4f}  RMSE: {rmse_:.4f}  R²: {r2_:.4f}')
    joblib.dump(m, MODEL_DIR / f'{model_name}_model.joblib')
print('Additional RF models saved.')

## 17. XGBoost

In [ ]:
print('Training XGBoost Regressor (Next_Day_Close)...')
xgb_reg = xgb.XGBRegressor(
    n_estimators=200 if not FAST_MODE else 50,
    max_depth=6, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, n_jobs=-1, random_state=RANDOM_STATE,
    early_stopping_rounds=20, verbosity=0
)
xgb_reg.fit(
    X_train, y_close_train,
    eval_set=[(X_val, y_close_val)],
    verbose=False
)
xgb_pred = xgb_reg.predict(X_test)

mae_xgb  = mean_absolute_error(y_close_test, xgb_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_close_test, xgb_pred))
r2_xgb   = r2_score(y_close_test, xgb_pred)
results['XGBoost'] = {'Task': 'Close', 'MAE': mae_xgb, 'RMSE': rmse_xgb, 'R2': r2_xgb}
print(f'  MAE: {mae_xgb:.4f}  RMSE: {rmse_xgb:.4f}  R²: {r2_xgb:.4f}')
joblib.dump(xgb_reg, MODEL_DIR / 'xgb_close_model.joblib')

print('\nTraining XGBoost Classifier (Next_Day_Direction)...')
xgb_clf = xgb.XGBClassifier(
    n_estimators=200 if not FAST_MODE else 50,
    max_depth=5, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, n_jobs=-1, random_state=RANDOM_STATE,
    early_stopping_rounds=20, verbosity=0, use_label_encoder=False,
    eval_metric='logloss'
)
xgb_clf.fit(
    X_train, y_dir_train,
    eval_set=[(X_val, y_dir_test[:len(y_dir_test)])],
    verbose=False
)
xgb_dir_pred = xgb_clf.predict(X_test)
xgb_dir_prob = xgb_clf.predict_proba(X_test)[:, 1]

acc_xgb  = accuracy_score(y_dir_test, xgb_dir_pred)
f1_xgb   = f1_score(y_dir_test, xgb_dir_pred, zero_division=0)
auc_xgb  = roc_auc_score(y_dir_test, xgb_dir_prob)
results['XGBoost Classifier'] = {'Task': 'Direction', 'Dir_Accuracy': acc_xgb,
                                   'Dir_F1': f1_xgb, 'ROC_AUC': auc_xgb}
print(f'  Accuracy: {acc_xgb:.4f}  F1: {f1_xgb:.4f}  AUC: {auc_xgb:.4f}')
joblib.dump(xgb_clf, MODEL_DIR / 'xgb_direction_model.joblib')
print('XGBoost models saved.')

## 18. LSTM (Optional)

LSTM is trained only if TensorFlow is available and the dataset has sufficient data. Due to computational constraints on a standard laptop, LSTM is trained on a single representative stock (RELIANCE.NS) with reduced epochs.

In [ ]:
LSTM_SEQ_LEN = 30
LSTM_TICKER  = 'RELIANCE.NS'

if LSTM_AVAILABLE and not FAST_MODE:
    print(f'Training LSTM on {LSTM_TICKER}...')
    s = df_ml[df_ml['Ticker'] == LSTM_TICKER].copy().sort_values('Date')
    s_feat = s[FEATURE_COLS_FINAL].fillna(0).values
    s_target = s['Next_Day_Close'].values

    # Scale
    lstm_feat_sc = StandardScaler()
    lstm_tgt_sc  = StandardScaler()
    s_feat_sc    = lstm_feat_sc.fit_transform(s_feat)
    s_tgt_sc     = lstm_tgt_sc.fit_transform(s_target.reshape(-1, 1)).ravel()

    # Build sequences
    def make_sequences(X, y, seq_len):
        Xs, ys = [], []
        for i in range(seq_len, len(X)):
            Xs.append(X[i-seq_len:i])
            ys.append(y[i])
        return np.array(Xs), np.array(ys)

    Xs, ys = make_sequences(s_feat_sc, s_tgt_sc, LSTM_SEQ_LEN)

    # Chronological split within this stock
    n = len(Xs)
    tr_e  = int(n * TRAIN_RATIO)
    val_e = int(n * (TRAIN_RATIO + VAL_RATIO))

    Xs_tr, ys_tr   = Xs[:tr_e], ys[:tr_e]
    Xs_val, ys_val = Xs[tr_e:val_e], ys[tr_e:val_e]
    Xs_te, ys_te   = Xs[val_e:], ys[val_e:]

    # Build model
    lstm_model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(LSTM_SEQ_LEN, Xs_tr.shape[2])),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(1)
    ])
    lstm_model.compile(optimizer='adam', loss='mse')

    es = EarlyStopping(patience=10, restore_best_weights=True)
    history = lstm_model.fit(
        Xs_tr, ys_tr,
        validation_data=(Xs_val, ys_val),
        epochs=50, batch_size=32, callbacks=[es], verbose=0
    )
    print(f'  LSTM trained for {len(history.history["loss"])} epochs.')

    lstm_pred_sc = lstm_model.predict(Xs_te, verbose=0).ravel()
    lstm_pred    = lstm_tgt_sc.inverse_transform(lstm_pred_sc.reshape(-1, 1)).ravel()
    ys_te_orig   = lstm_tgt_sc.inverse_transform(ys_te.reshape(-1, 1)).ravel()

    mae_lstm  = mean_absolute_error(ys_te_orig, lstm_pred)
    rmse_lstm = np.sqrt(mean_squared_error(ys_te_orig, lstm_pred))
    r2_lstm   = r2_score(ys_te_orig, lstm_pred)
    results['LSTM'] = {'Task': f'Close ({LSTM_TICKER})', 'MAE': mae_lstm,
                        'RMSE': rmse_lstm, 'R2': r2_lstm}
    print(f'  MAE: {mae_lstm:.4f}  RMSE: {rmse_lstm:.4f}  R²: {r2_lstm:.4f}')
    lstm_model.save(str(MODEL_DIR / 'lstm_close_model.h5'))
    joblib.dump(lstm_feat_sc, SCALER_DIR / 'lstm_feat_scaler.joblib')
    joblib.dump(lstm_tgt_sc,  SCALER_DIR / 'lstm_tgt_scaler.joblib')
    print('LSTM model saved.')
elif FAST_MODE:
    print('LSTM skipped: FAST_MODE is enabled.')
else:
    print('LSTM skipped: TensorFlow not installed.')
    print('To enable LSTM, install TensorFlow: pip install tensorflow')

## 19. Model Evaluation — Central Comparison Table

In [ ]:
comparison_df = pd.DataFrame(results).T.reset_index()
comparison_df.rename(columns={'index': 'Model'}, inplace=True)
print('=' * 80)
print('CENTRAL MODEL COMPARISON TABLE')
print('=' * 80)
display(comparison_df)

# Save metrics
comparison_df.to_csv(METRICS_DIR / 'model_comparison.csv', index=False)
print(f'\nMetrics saved to: {METRICS_DIR}/model_comparison.csv')

# Plot comparison
close_models = comparison_df[comparison_df['Task'] == 'Close'].dropna(subset=['R2'])
if not close_models.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, metric in zip(axes, ['MAE', 'RMSE', 'R2']):
        vals = close_models[metric].astype(float)
        bars = ax.bar(close_models['Model'], vals, color='steelblue', edgecolor='white')
        ax.set_title(f'{metric} — Next-Day Close', fontsize=12)
        ax.set_ylabel(metric)
        ax.set_xticklabels(close_models['Model'], rotation=25, ha='right')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    plt.suptitle('Model Comparison — Next-Day Close Price', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'model_comparison_close.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# Actual vs Predicted — Close (XGBoost)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Time series
axes[0].plot(test_df['Date'].values[:500], y_close_test.values[:500],
             color='navy', linewidth=1, label='Actual', alpha=0.8)
axes[0].plot(test_df['Date'].values[:500], xgb_pred[:500],
             color='orange', linewidth=1, label='XGBoost Predicted', alpha=0.8)
axes[0].set_title('Actual vs XGBoost Predicted — Next-Day Close (first 500 test points)', fontsize=12)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Price (₹)')
axes[0].legend()

# Scatter
axes[1].scatter(y_close_test.values[:2000], xgb_pred[:2000],
                alpha=0.2, color='steelblue', s=10)
mn, mx = y_close_test.values[:2000].min(), y_close_test.values[:2000].max()
axes[1].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect prediction')
axes[1].set_title('Scatter: Actual vs Predicted Close', fontsize=12)
axes[1].set_xlabel('Actual Close')
axes[1].set_ylabel('Predicted Close')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'actual_vs_predicted_close.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Direction classification evaluation
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Confusion Matrix — XGBoost
cm = confusion_matrix(y_dir_test, xgb_dir_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['DOWN/FLAT', 'UP'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('XGBoost — Confusion Matrix (Direction)', fontsize=12)

# ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_dir_test, xgb_dir_prob)
axes[1].plot(fpr, tpr, color='steelblue', label=f'XGBoost (AUC={auc_xgb:.3f})')
fpr_rf, tpr_rf, _ = roc_curve(y_dir_test, rf_dir_prob)
axes[1].plot(fpr_rf, tpr_rf, color='orange', label=f'RF (AUC={auc_rf:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random')
axes[1].set_title('ROC Curve — Next-Day Direction', fontsize=12)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'classification_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nXGBoost Classification Report (Direction):')
print(classification_report(y_dir_test, xgb_dir_pred, target_names=['DOWN/FLAT', 'UP']))

## 20. Model Explainability

Feature importance reveals which features the models rely on. This does not imply causal relationships.

In [ ]:
# XGBoost feature importance
xgb_fi = pd.Series(xgb_reg.feature_importances_, index=FEATURE_COLS_FINAL)
xgb_fi_top20 = xgb_fi.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(14, 7))
xgb_fi_top20.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('XGBoost — Top 20 Feature Importances (Next-Day Close)', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_importance_xgb.png', dpi=120, bbox_inches='tight')
plt.show()

# Random Forest feature importance
rf_fi = pd.Series(rf_reg.feature_importances_, index=FEATURE_COLS_FINAL)
rf_fi_top20 = rf_fi.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(14, 7))
rf_fi_top20.sort_values().plot(kind='barh', ax=ax, color='teal', edgecolor='white')
ax.set_title('Random Forest — Top 20 Feature Importances (Next-Day Close)', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_importance_rf.png', dpi=120, bbox_inches='tight')
plt.show()

print('Top 10 features (XGBoost):')
print(xgb_fi_top20.head(10).to_string())

## 21. Prediction Demonstration

In [ ]:
def predict_stock(ticker, df_source, model_close, model_dir, feature_cols):
    """Predict next-day close and direction for the latest available data of a ticker."""
    s = df_source[df_source['Ticker'] == ticker].sort_values('Date')
    if s.empty:
        print(f'No data for {ticker}')
        return
    latest = s.iloc[-1]
    X = latest[feature_cols].fillna(0).values.reshape(1, -1)
    pred_close = model_close.predict(X)[0]
    pred_dir   = model_dir.predict(X)[0]
    pred_prob  = model_dir.predict_proba(X)[0, 1]
    actual     = latest['Close']

    print(f'\n=== Prediction for {ticker} (as of {latest["Date"].date()}) ===')
    print(f'  Current Close:       ₹{actual:.2f}')
    print(f'  Predicted Next Close: ₹{pred_close:.2f}')
    print(f'  Expected Change:      ₹{pred_close - actual:+.2f} ({(pred_close/actual - 1)*100:+.2f}%)')
    print(f'  Direction:            {"UP ↑" if pred_dir == 1 else "DOWN/FLAT ↓"} (prob={pred_prob:.3f})')
    print('  ⚠ Predictions are model estimates only — not financial advice.')

for tk in ['RELIANCE.NS', 'INFY.NS', 'HDFCBANK.NS']:
    if tk in df_ml['Ticker'].values:
        predict_stock(tk, df_ml, xgb_reg, xgb_clf, FEATURE_COLS_FINAL)

## 22. Additional Visualizations

In [ ]:
# Residual plot
residuals = y_close_test.values - xgb_pred
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].scatter(xgb_pred[:3000], residuals[:3000], alpha=0.15, color='steelblue', s=8)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_title('Residuals vs Predicted (XGBoost Close)', fontsize=12)
axes[0].set_xlabel('Predicted Close')
axes[0].set_ylabel('Residual')

axes[1].hist(residuals, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Prediction Error Distribution (XGBoost Close)', fontsize=12)
axes[1].set_xlabel('Residual (Actual − Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'residual_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Rolling volatility across sectors
sector_vol = df.groupby(['Year', 'Sector'])['Daily_Return'].std().reset_index()
sector_vol.rename(columns={'Daily_Return': 'Vol'}, inplace=True)
sector_vol['Annualized_Vol_%'] = sector_vol['Vol'] * np.sqrt(252) * 100

pivot = sector_vol.pivot(index='Year', columns='Sector', values='Annualized_Vol_%')
fig, ax = plt.subplots(figsize=(18, 7))
pivot.plot(ax=ax, linewidth=1.5, alpha=0.85)
ax.set_title('Annualized Volatility by Sector — Yearly', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Annualized Volatility (%)')
ax.legend(loc='upper right', fontsize=8, ncol=3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sector_volatility_yearly.png', dpi=120, bbox_inches='tight')
plt.show()

## 23. Findings

Key findings from this analysis:

1. **Market growth**: The Nifty 50 has delivered substantial long-term returns, with median stock prices growing significantly from 1999 to 2026.
2. **Sector performance**: IT, Financials, and Consumer Durables show strong long-term returns; Metals and Energy exhibit higher volatility.
3. **Seasonal patterns**: Some months show consistent return tendencies, but the signal is noisy and not reliable for trading.
4. **Model performance**: Tree-based models (XGBoost, Random Forest) outperform linear regression for next-day price prediction due to non-linear relationships. LSTM can capture temporal patterns but requires careful tuning.
5. **Direction prediction**: The classification task is harder than regression — accuracy around the 55–60% range is typical and does not guarantee profitable trading after transaction costs.
6. **Leakage prevention**: Chronological splitting and excluding future-containing features (52-week high/low) are critical for fair evaluation.
7. **Feature importance**: Recent price levels and momentum features (Close_Lag1, SMA_20, ATR) dominate model decisions.

## 24. Limitations

- **Survivorship bias**: Only current Nifty 50 constituents are included. Delisted companies are absent.
- **Static fundamentals**: PE ratio, Market Cap and similar columns appear to reflect a snapshot rather than historical time-series values.
- **Indian holidays**: Business-day calculations underestimate missing trading days without an India-specific holiday calendar.
- **Market regime shifts**: Models trained on historical data may not generalize after major structural breaks (COVID-19, policy changes).
- **No news/sentiment data**: Macroeconomic and sentiment signals are not included.
- **LSTM compute cost**: Full LSTM training across all stocks requires GPU/cloud compute beyond a standard laptop.

## 25. Conclusion

This project demonstrates a complete, reproducible end-to-end data analytics and machine learning pipeline for Nifty 50 stock market data. The multi-model approach provides comparative insight into the strengths and weaknesses of different algorithms for financial time-series prediction. The Streamlit dashboard (`dashboard/app.py`) makes all findings accessible to non-technical users.

**Important reminder:** All predictions are statistical estimates based on historical patterns. This project is for educational purposes only and does not constitute financial advice. Consult qualified financial professionals before making investment decisions.

---
*IBM SkillsBuild Data Analytics with AI Academy Internship Program — Bharat Care × AICTE*

In [ ]:
# Save feature column list and metadata for dashboard
meta_out = {
    'feature_cols': FEATURE_COLS_FINAL,
    'tickers': sorted(df['Ticker'].unique().tolist()),
    'sectors': sorted(df['Sector'].unique().tolist()),
    'date_min': str(df['Date'].min().date()),
    'date_max': str(df['Date'].max().date()),
    'n_rows': len(df),
    'random_state': RANDOM_STATE,
    'train_cut': str(train_cut.date()),
    'val_cut': str(val_cut.date()),
    'lstm_seq_len': LSTM_SEQ_LEN,
}
with open(METRICS_DIR / 'pipeline_meta.json', 'w') as f:
    json.dump(meta_out, f, indent=2)
print('Pipeline metadata saved.')
print('\n✅ Notebook execution complete!')